In [0]:
# Questo notebook crea la prima tabella Gold del progetto.
#
# Legge la Silver meteo consolidata:
# progetto_meteo.silver.dati_aggiornati
#
# Il notebook aggrega questi dati a livello di:
# - Area
# - Data
# - Ora
#
# In questo modo passa dal dettaglio delle città alla macroarea finale del progetto.
#
# Il risultato viene salvato nella tabella Gold:
# progetto_meteo.gold.dati_aggregati
#
# Questa tabella contiene solo la parte meteo aggregata.
# Verrà poi usata da Gold_Dati_Studio per fare il join con i dati energia
# presenti in silver.energy_block.
#
# Logica di aggregazione:
# - Temperatura = media di Temp_Oraria
# - Umidita = media di Umidita
# - Velocita_Vento = media di Velocita_Vento
# - Precipitazioni = media di Precipitazioni
#
# La colonna Data viene convertita in stringa formato dd/MM/yyyy,
# così resta coerente con energy_block e con la tabella finale dati_studio.

from pyspark.sql import functions as F


# Imposto catalogo e tabelle del progetto.
catalogo = "progetto_meteo"

tabella_dati_aggiornati = f"{catalogo}.silver.dati_aggiornati"
tabella_dati_aggregati = f"{catalogo}.gold.dati_aggregati"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo i dati meteo già consolidati in Silver.
# Questa tabella contiene lo storico iniziale più i giorni aggiornati dal Patcher.
df_meteo = spark.table(tabella_dati_aggiornati)


# Controllo che la Silver abbia dati.
# Se la tabella è vuota, interrompo il notebook perché non si può creare la Gold meteo.
righe_silver = df_meteo.count()

if righe_silver == 0:
    raise Exception(f"Nessun dato trovato in {tabella_dati_aggiornati}. Esegui prima Bootstrapper/Clonatore o Patcher.")


# Aggrego i dati meteo per Area, Data e Ora.
# Qui passo dal livello città al livello macroarea.
df_dati_aggregati = (
    df_meteo
    .groupBy(
        "Area",
        "Data",
        "Ora"
    )
    .agg(
        F.round(F.avg("Temp_Oraria"), 2).alias("Temperatura"),
        F.round(F.avg("Umidita"), 2).alias("Umidita"),
        F.round(F.avg("Velocita_Vento"), 2).alias("Velocita_Vento"),
        F.round(F.avg("Precipitazioni"), 2).alias("Precipitazioni")
    )
    .withColumn("Data", F.date_format("Data", "dd/MM/yyyy"))
    .select(
        "Area",
        "Data",
        "Ora",
        "Temperatura",
        "Umidita",
        "Velocita_Vento",
        "Precipitazioni"
    )
)


# Controllo che l'aggregazione abbia prodotto righe.
# Se non ci sono righe aggregate, non aggiorno la Gold.
righe_aggregate = df_dati_aggregati.count()

if righe_aggregate == 0:
    raise Exception("Aggregazione meteo completata, ma nessuna riga prodotta. Non aggiorno la Gold.")


# Salvo il risultato nella tabella Gold dati_aggregati.
# Uso overwrite perché questa tabella deve essere sempre coerente
# con l'intero contenuto della Silver meteo.
df_dati_aggregati.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    tabella_dati_aggregati
)


# Controllo finale.
# Mostro un campione ordinato della Gold meteo appena creata.
display(
    spark.table(tabella_dati_aggregati)
    .orderBy("Area", "Data", "Ora")
    .limit(100)
)


# Stampo un riepilogo finale della creazione Gold meteo.
print(f"Dati aggregati completati. Tabella aggiornata: {tabella_dati_aggregati}")
print(f"Righe Silver lette: {righe_silver}")
print(f"Righe salvate in dati_aggregati: {spark.table(tabella_dati_aggregati).count()}")